### BPE Tokenization from Scratch

BPE Tokenization Algo (Simplified):

- Input Raw Text: "This text is raw text"

- Convert raw text to word freq dict: {"This": 1, "text": 2, "is": 1, "raw": 1} and then convert the words in word freq dict into individual characters spaced along with a special word boundary notifier: {"T h i s  \</w>": 1, "t e x t  \</w>": 2, "i s  \</w>": 1, "r a w  \</w>": 1}

- Count bigram pairs and return a bigram pair freq dict: {(T, h): 1, (i, s): 2, ...}

- Find max pair in bigram pairs: e.g. (i, s)

- Merge (i, s) into a single entity (is): e.g post merge {"T h is </w>": 1, "t e x t </w>": 2, "is </w>": 1, "r a w </w>": 1}. Notice that we replaced 'i s' with 'is' and add this merging to a merge_rules file.

- Repeat "n_merges" times. "n_merges" is the only hyperparam in this alogrithm.

- Post n_merges, save all the merge pairs in exact order as they were performed in a file called merge_rules.txt. It's a simple text file with each line containing the symbols that were merged, in sequence.

- Also, when training is complete i.e post n_merges, we also store the final vocab with all the merged tokens as well as each individual character that was part of the training corpus into a dictionary and assign simple numbers to them which become their token IDs. Eg. Training corpus "This text is raw text" -> Vocab: {'\[UNK\]': 0, '\[PAD\]': 1, '\[EOS\]': 2, '\[BOS\]': 3, 'T': 4, 'h': 5, 'i': 6, 's': 7, 't': 8, 'e': 9, 'x': 10, 'r': 11, 'a': 12, 'w': 13, 'es': 14, 'tex': 15, ...}

**Note:** Usually some special tokens including but not limited to [PAD], [UNK], [EOS], [BOS] etc. are also defined though they are not necessarily part of core BPE but aid tokenization and are kept with lowest token IDs so that they can be easily grouped out, as well their positions always remain fixed, even if the vocab grows.

**Note:** It's upto you whether you'd want your tokenizer to be case-sensitive i.e. 'T', 't' will be two separate tokens and how you'd like to handle punctuation. Our implementation of BPE is **case-sensitive** by default and we'll handle **punctuation the naive way without handling abbreviations like P.S or don't**.

BPE Implementation:

- We will start with 3 functions: 
    - one that takes in raw text and gives us char spaced (with word boundaries) freq dict, 
    - second that takes in char spaced word freq dict and gives us bigram pairs freq dict and, 
    - third that performs the merge
- A helper function to save merge rules and a function to store the final vocab post training.

In [31]:
import os
import json
import string
from functools import reduce
from collections import defaultdict, Counter

In [32]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [33]:
def char_spaced_word_freq(raw_text):
    # To naively handle punctuation we will just treat each punctuation occurrence as an individual word.
    # To that end, we'll simply replace any punctuation with a space before and after it.
    raw_text = reduce(lambda text, punc: text.replace(punc, f" {punc} "), string.punctuation, raw_text)
    # Convert raw text into character spaced words with </w> boundary notifier
    words = []
    s = ''
    for char in raw_text:
        if char == ' ' and s!='':
            s+='</w>'
            words.append(s)
            s = ''
        else:
            s += char + ' '
    if s!='':
        s+='</w>'
        words.append(s)

    # Count word frequency
    freq = Counter(words)

    return dict(freq)

In [34]:
def bigram_pair_freq(char_spaced_freq):
    bigram_freq_count = defaultdict(int)
    for word, freq in char_spaced_freq.items():
        # Convert 'T h i s </w>' to [T, h, i, s]
        symbols = word.split()
        # Loop through the symbols and count bigram pairs
        for i in range(len(symbols)-1):
            bigram_freq_count[symbols[i], symbols[i+1]] += freq
    return dict(bigram_freq_count)

In [35]:
def merge(bigram_pair, char_spaced_freq):
    # Need to search 'x y' substring in our char spaced freq key
    pair = ' '.join(bigram_pair)
    # Replacement to replace the bigram pair with
    replacement = ''.join(bigram_pair)

    merged_char_spaced_freq = defaultdict(int)
    for word, freq in char_spaced_freq.items():
        if pair in word:
            new_word = word.replace(pair, replacement)
            merged_char_spaced_freq[new_word] = freq
        else:
            merged_char_spaced_freq[word] = freq
    
    return dict(merged_char_spaced_freq)

In [36]:
def save_merge_rules(merge_rules, path="merge_rules.txt"):
    if os.path.exists(path):
        with open(path, "a+") as f:
            for merge_pair in merge_rules:
                f.write(f"{merge_pair[0]} {merge_pair[1]}\n")
    else:
        with open(path, "w") as f:
            f.write("#version: 0.1\n")
            for merge_pair in merge_rules:
                f.write(f"{merge_pair[0]} {merge_pair[1]}\n")

### Vocab

- Our vocab consists of all the base chars in our input sequence + special tokens + all the merged pairs from merge rules.

**Initial Vocab**

- Our initial vocab will consist of all the characters in our input text + word boundary char </w> as well as any special token, we would want to keep [UNK], [PAD], [EOS], [BOS] etc.

- One thing to note is that, we have used \</w> as word boundary token instead of usual whitespace character, as it was represented in the original [Paper](https://arxiv.org/abs/1508.07909)

In [37]:
def build_vocab(input_text, merge_rules, special_tokens = ['[UNK]', '[PAD]', '[BOS]', '[EOS]']):
    input_chars = sorted(list(set(input_text)- {' '}))
    input_chars.append('</w>')
    vocab = {token:i for i, token in enumerate(special_tokens+input_chars+list(map(''.join, merge_rules)))}
    return vocab

def save_vocab(vocab, path="vocab.json"):
    with open(path, "w") as f:
        f.write(json.dumps(vocab))

### BPE in action

- This phase is what we call training.

In [38]:
input_text = "don't stop, it's U.S.A. territory"
merge_rules = []

In [39]:
# Create char spaced frequency dict with word boundaries
char_spaced_freq = char_spaced_word_freq(input_text)
print(f"Input: {input_text}")
print(f"Char Spaced Freq. Dict: {char_spaced_freq}")

Input: don't stop, it's U.S.A. territory
Char Spaced Freq. Dict: {'d o n </w>': 1, "' </w>": 2, 't </w>': 1, 's t o p </w>': 1, ', </w>': 1, '  i t </w>': 1, 's </w>': 1, 'U </w>': 1, '. </w>': 3, 'S </w>': 1, 'A </w>': 1, '  t e r r i t o r y </w>': 1}


In [40]:
# Create bigram pairs freq dict
bigram_pairs = bigram_pair_freq(char_spaced_freq)
print(f"Bigram Pairs Freq. Dict: {bigram_pairs}")

Bigram Pairs Freq. Dict: {('d', 'o'): 1, ('o', 'n'): 1, ('n', '</w>'): 1, ("'", '</w>'): 2, ('t', '</w>'): 2, ('s', 't'): 1, ('t', 'o'): 2, ('o', 'p'): 1, ('p', '</w>'): 1, (',', '</w>'): 1, ('i', 't'): 2, ('s', '</w>'): 1, ('U', '</w>'): 1, ('.', '</w>'): 3, ('S', '</w>'): 1, ('A', '</w>'): 1, ('t', 'e'): 1, ('e', 'r'): 1, ('r', 'r'): 1, ('r', 'i'): 1, ('o', 'r'): 1, ('r', 'y'): 1, ('y', '</w>'): 1}


In [41]:
# Find the bigram pair with max freq
max_freq_bigram = max(bigram_pairs, key=bigram_pairs.get)
print(f"Most frequent bigram pair: {max_freq_bigram}")

Most frequent bigram pair: ('.', '</w>')


In [42]:
# Merge the most frequent bigram pair and update the char spaced dict
new_char_spaced_dict = merge(max_freq_bigram, char_spaced_freq)
print(f"Before merge: {char_spaced_freq}")
print(f"After merge: {new_char_spaced_dict}")

Before merge: {'d o n </w>': 1, "' </w>": 2, 't </w>': 1, 's t o p </w>': 1, ', </w>': 1, '  i t </w>': 1, 's </w>': 1, 'U </w>': 1, '. </w>': 3, 'S </w>': 1, 'A </w>': 1, '  t e r r i t o r y </w>': 1}
After merge: {'d o n </w>': 1, "' </w>": 2, 't </w>': 1, 's t o p </w>': 1, ', </w>': 1, '  i t </w>': 1, 's </w>': 1, 'U </w>': 1, '.</w>': 3, 'S </w>': 1, 'A </w>': 1, '  t e r r i t o r y </w>': 1}


In [43]:
# Store the merge rules for saving to disk
merge_rules.append(max_freq_bigram)
print(f"Merge rules: {merge_rules}")

Merge rules: [('.', '</w>')]


### Full training loop for an input, for n_merges

- Training BPE for n_merges on an input corpus and saving merge_rules and vocab.

In [44]:
n_merges = 5
input_text = "This text is raw text. don't stop, it's U.S.A. territory"
merge_rules = []
# Create char spaced frequency dict with word boundaries
char_spaced_freq = char_spaced_word_freq(input_text)
print(f"Input: {input_text}")
print(f"Initial Char Spaced Freq. Dict: {char_spaced_freq}")

for i in range(n_merges):
    print("*"*50)
    print(f"Iteration {i}:")
    # Create bigram pairs freq dict
    bigram_pairs = bigram_pair_freq(char_spaced_freq)
    print(f"Bigram Pairs Freq. Dict: {bigram_pairs}")
    # Find the bigram pair with max freq
    max_freq_bigram = max(bigram_pairs, key=bigram_pairs.get)
    print(f"Most frequent bigram pair: {max_freq_bigram}")
    
    # Merge the most frequent bigram pair and update the char spaced dict
    print(f"Before merge: {char_spaced_freq}")
    char_spaced_freq = merge(max_freq_bigram, char_spaced_freq)
    print(f"After merge: {char_spaced_freq}")

    # Store the merge rules for saving to disk
    merge_rules.append(max_freq_bigram)
    print(f"Merge rules: {merge_rules}")

vocab = build_vocab(input_text, merge_rules)
# Save merge rules to disk
save_merge_rules(merge_rules)
# Save vocab to disk
save_vocab(vocab)
print("-"*50)
print(f"Final Merge Rules: {merge_rules}")
print(f"Vocab: {vocab}")
print("-"*50)

Input: This text is raw text. don't stop, it's U.S.A. territory
Initial Char Spaced Freq. Dict: {'T h i s </w>': 1, 't e x t </w>': 2, 'i s </w>': 1, 'r a w </w>': 1, '. </w>': 4, '  d o n </w>': 1, "' </w>": 2, 't </w>': 1, 's t o p </w>': 1, ', </w>': 1, '  i t </w>': 1, 's </w>': 1, 'U </w>': 1, 'S </w>': 1, 'A </w>': 1, '  t e r r i t o r y </w>': 1}
**************************************************
Iteration 0:
Bigram Pairs Freq. Dict: {('T', 'h'): 1, ('h', 'i'): 1, ('i', 's'): 2, ('s', '</w>'): 3, ('t', 'e'): 3, ('e', 'x'): 2, ('x', 't'): 2, ('t', '</w>'): 4, ('r', 'a'): 1, ('a', 'w'): 1, ('w', '</w>'): 1, ('.', '</w>'): 4, ('d', 'o'): 1, ('o', 'n'): 1, ('n', '</w>'): 1, ("'", '</w>'): 2, ('s', 't'): 1, ('t', 'o'): 2, ('o', 'p'): 1, ('p', '</w>'): 1, (',', '</w>'): 1, ('i', 't'): 2, ('U', '</w>'): 1, ('S', '</w>'): 1, ('A', '</w>'): 1, ('e', 'r'): 1, ('r', 'r'): 1, ('r', 'i'): 1, ('o', 'r'): 1, ('r', 'y'): 1, ('y', '</w>'): 1}
Most frequent bigram pair: ('t', '</w>')
Before merg

### Encode

- Now that a naive training of our tokenizer is complete and we have our vocab and merge rules, the next step is **encode()**.

- Encode is the method that converts our raw input text to token IDs as we see in real world implementations.

- Here's how encode will work in our case:

    - Take raw text as input
    
    - Run the pre-processing step of converting raw text into char spaced words (with word boundary in our case since we trained our tokenizer so and replaced ' ' whitespace chat with \</w> as per the original paper) and the only change is, instead of a freq dict, we will have a list of char spaced strings. So: "This word" gets converted to **[ 'T h i s \</w>', 'w o r d \</w>' ]**
    
    - Then we read our merge_rules.txt, that we saved during training, and iterate and apply merge_rules, in order, to each word.
    
    - Now that we have our char spaced input converted to what we would call tokens (since we applied rules from our merge_rules that we prepared during training), we then assign IDs to those tokens based on our vocab.json that we saved during training. If a token does not exist in vocab.json, we assign it the ID of our \[UNK\] token. This gracefully handles out of vocab scenarios. 

In [45]:
with open('merge_rules.txt', 'r') as f:
    merge_rules = [line.strip() for line in f.readlines()]
    merge_rules.pop(0)

with open('vocab.json', 'r') as f:
    vocab = json.loads(f.read())
    reverse_vocab = {v:k for k, v in vocab.items()}

In [46]:
def __preprocess(raw_text):
    words = raw_text.split(' ')
    preprocessed = []
    for word in words:
        tmp = []
        for char in word:
            tmp.append(char)
        tmp.append('</w>')
        preprocessed.append(tmp)
    preprocessed = list(map(' '.join, preprocessed))
    return preprocessed 

def encode(input_text, vocab, merge_rules):
    preprocessed_text = __preprocess(input_text)

    for pattern in merge_rules:
        replacement = ''.join(pattern.split(' '))
        for i, entry in enumerate(preprocessed_text):
            new = entry.replace(pattern, replacement)
            preprocessed_text[i] = new
    
    # Now, each word inside preprocessed_text has been converted into tokens.
    # Convert into a flattened list of tokens
    tokens = [token for processed_word in preprocessed_text for token in processed_word.split(' ')]
    tokenIDs = [vocab.get(i, vocab['[UNK]']) for i in tokens]

    return tokens, tokenIDs

tokens, tokenIDs = encode("This text is raw text", vocab, merge_rules)

### Decode

- The final step in our tokenizer implementation is **decode()**.

- Decode is simply getting tokenIDs and returning actual tokens for them i.e. reverse lookup of vocab. Simple as that.

- Additionally, decode also entails a small post processing step to reconstruct the actual words from tokens i.e just joining them till word boundary \</w> in our case.

In [47]:
def decode(tokenIDs, reverse_vocab):
    tokens = []
    for id in tokenIDs:
        tokens.append(reverse_vocab.get(id, reverse_vocab[0]))
    words = list(map(str.strip, (''.join(tokens)).split("</w>")))
    return tokens, words        

In [48]:
tokens_d, words = decode(tokenIDs, reverse_vocab)

In [49]:
tokens == tokens_d

True